# 04 — Balanceamento do treino binário (RF e SVM)

Este notebook **não treina nada**. Ele monta, a partir da view definitiva do notebook 03, o **conjunto de treino balanceado** (mesma quantidade de BENIGN e de ataque) e uma **validação separada**, para o RF e o SVM binários.

## Decisões (e por quê)

| Pergunta | Decisão |
|---|---|
| **Quantas linhas?** | O limite real é o número de BENIGN no treino: **50.768**. Com **80%** deles no ajuste do modelo: **40.614 BENIGN + 40.614 ataques = 81.228 linhas**. "100 mil" foi um exemplo de ordem de grandeza: 50 mil de cada usaria quase todos os BENIGN e não sobraria nenhum para validar. É ajustável (`FRACAO_BENIGN_TREINO`: 0,90 daria ~91 mil) |
| **Quem recebe esse conjunto?** | **O mesmo** para RF e SVM, para a comparação ser justa |
| **Como escolher os ataques?** | **Sorteio aleatório dentro de cada tipo de ataque**, e não as primeiras linhas e nem um sorteio puro. Um sorteio puro seria dominado pelos tipos maiores (milhões de linhas), e os pequenos quase sumiriam. Por padrão, cada tipo entra com a **mesma quantidade** (`ALPHA_ATAQUES = 0`); tipos pequenos entram inteiros (menos a parte da validação) e o que sobra é repartido entre os outros |
| **E o teste?** | **Não é balanceado.** O teste deve refletir a proporção real (~0,8% BENIGN); balancear o teste mudaria o que as métricas significam. Usa-se a view `prep_vw_rf_bin_teste_sem_sobrep`, como está |
| **Validação** | Os BENIGN e ataques que **não** entraram no ajuste formam a validação (para escolher `C`, `gamma`, limiar etc.). Os ataques da validação são uma **amostra por tipo com pesos**, que permitem recompor a proporção real nas métricas. O treino final com todos os BENIGN (o *refit*) fica para o notebook de treino |

## O que é criado (nada existente é alterado)

| Objeto | Conteúdo |
|---|---|
| `prep_bal_bin_selecao` | Quais linhas foram escolhidas, com o papel (`treino` ou `validacao`), o tipo de ataque e o peso. É a **auditoria** da seleção |
| `prep_bal_bin_dados` | Os dados dessas linhas, copiados da view definitiva (nulos já pela mediana, sem as colunas de AUC alto). Tabela pequena: leitura rápida e seleção congelada |
| `prep_vw_bal_bin_treino` | `id_linha`, `y` e atributos do conjunto balanceado de treino |
| `prep_vw_bal_bin_validacao` | Igual, mais a coluna `peso` |

O tipo de ataque (`tipo`) fica só na tabela de auditoria e **nunca** nas views de treino: não pode ser atributo do modelo.

## Antes de rodar
O notebook 03 precisa ter criado `prep_vw_rf_bin_treino`. A conexão é a mesma do 02 e do 03 (arquivo `.env`): coloque este arquivo na **mesma pasta**.

In [10]:
import os, re, json, time, http.client, urllib.parse
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 200); pd.set_option("display.max_rows", 200); pd.set_option("display.max_colwidth", 120)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


def load_env(path: Path) -> dict:
    values = {}
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


env_path = REPO_ROOT / ".env"
if not env_path.exists():
    raise FileNotFoundError(f"Arquivo .env não encontrado em: {env_path}")
env = {**load_env(env_path), **os.environ}

CH_HOST, CH_PORT = env["CLICKHOUSE_HOST"], env["CLICKHOUSE_PORT"]
CH_USER, CH_PASSWORD, CH_DATABASE = env["CLICKHOUSE_USER"], env["CLICKHOUSE_PASSWORD"], env["CLICKHOUSE_DATABASE"]
CH_SCHEME = env.get("CLICKHOUSE_SCHEME", "https")
HTTP_TIMEOUT = 1800
print(f"ClickHouse: {CH_HOST}:{CH_PORT}/{CH_DATABASE}  |  usuário: {CH_USER}")

ClickHouse: clickhouse-production-e204.up.railway.app:443/railway  |  usuário: clickhouse


In [11]:
# ---------------- transporte HTTP direto (conexão reutilizada + reconexão automática só para leituras) ----------------
class ErroConexao(Exception):
    pass


CONEXAO_ESPERAS = (0, 5, 15, 45, 90)
_conn = None
ULT_STATS = None                  # estatísticas (linhas lidas etc.) da última consulta JSON


def _fechar_conexao():
    global _conn
    try:
        if _conn is not None:
            _conn.close()
    except Exception:
        pass
    _conn = None


def _http(query: str) -> pd.DataFrame:
    """Executa via HTTP. Repetir leituras é seguro; escritas não são reenviadas aqui (quem repete é carregar(), com verificação de contagem)."""
    global _conn, ULT_STATS
    params = {"database": CH_DATABASE, "default_format": "JSONCompact", "wait_end_of_query": 1}
    primeiro = re.match(r"\s*(\w+)", query)
    verbo = primeiro.group(1).upper() if primeiro else ""
    if verbo == "ALTER":
        params["mutations_sync"] = 2          # espera o DELETE terminar antes de responder
    caminho = "/?" + urllib.parse.urlencode(params)
    cab = {"X-ClickHouse-User": CH_USER, "X-ClickHouse-Key": CH_PASSWORD, "Connection": "keep-alive"}
    somente_leitura = verbo not in {"INSERT", "CREATE", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE", "DELETE", "UPDATE"}
    tentativas = CONEXAO_ESPERAS if somente_leitura else (0,)
    ultimo = None
    for espera in tentativas:
        if espera:
            print(f"  conexão instável ({type(ultimo).__name__}: {str(ultimo)[:160]}); nova tentativa em {espera}s...")
            time.sleep(espera)
        try:
            if _conn is None:
                cls = http.client.HTTPSConnection if CH_SCHEME == "https" else http.client.HTTPConnection
                _conn = cls(CH_HOST, int(CH_PORT), timeout=HTTP_TIMEOUT)
            _conn.request("POST", caminho, body=query.encode("utf-8"), headers=cab)
            resp = _conn.getresponse()
            corpo = resp.read()
        except (OSError, http.client.HTTPException) as e:
            ultimo = e
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"Falha de conexão durante escrita: {e!r}") from e
            continue
        if resp.status in (502, 503, 504) or (resp.status >= 500 and not corpo.startswith(b"Code:")):
            ultimo = Exception(f"HTTP {resp.status} [código ClickHouse: {resp.getheader('X-ClickHouse-Exception-Code')}] corpo={corpo[:300].decode('utf-8', 'replace')!r}")
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"HTTP {resp.status} durante escrita. Resposta: {corpo.decode('utf-8', 'replace')[:500]!r}")
            continue
        if resp.status != 200:
            raise Exception(corpo.decode("utf-8", "replace")[:2000])
        if not corpo.strip():
            return pd.DataFrame()
        dados = json.loads(corpo)
        ULT_STATS = dados.get("statistics")
        return pd.DataFrame(dados.get("data", []), columns=[m["name"] for m in dados.get("meta", [])])
    raise ErroConexao(f"Sem conexão com o ClickHouse após {len(tentativas)} tentativas: {ultimo!r}")

In [12]:
# ---------------- TRAVA: só é possível escrever em tabelas prep_* ----------------
PREFIXO_NOVO = "prep_"          # tabelas que ESTE notebook pode criar / alimentar / apagar

_LITERAL = re.compile(r"'[^'\\]*(?:\\.[^'\\]*)*'")      # literais entre aspas simples (aceita \' escapado)


def _sem_literais(s: str) -> str:
    # texto SQL sem o conteúdo dos literais: ';' ou 'DROP TABLE' dentro de aspas NÃO contam na checagem da trava
    return _LITERAL.sub("''", s)


_VERBOS_ESCRITA = ("CREATE", "INSERT", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE")
_VERBOS_PROIBIDOS = ("DELETE", "UPDATE", "ATTACH", "DETACH", "EXCHANGE", "GRANT", "REVOKE", "KILL", "SYSTEM", "SET", "USE")
_ALVO = re.compile(r"\b(?:TABLE|INTO|VIEW)\s+(?:IF\s+(?:NOT\s+)?EXISTS\s+)?(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)
_RENAME = re.compile(r"(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)\s+TO\s+(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)


def _tabela_de(ident: str) -> str:
    # 'banco.tabela' ou '`tabela`' -> 'tabela' (e exige que o banco, se informado, seja o atual)
    partes = [p.strip("`") for p in re.findall(r"`[^`]+`|[^.]+", ident)]
    if len(partes) == 2 and partes[0] != CH_DATABASE:
        raise PermissionError(f"Escrita bloqueada: banco '{partes[0]}' diferente do atual '{CH_DATABASE}'.")
    return partes[-1]


def validar_escrita(sql: str, permitir_prefixos=()) -> None:
    # Levanta PermissionError se o comando escreveria fora das tabelas permitidas. Leituras passam sem restrição.
    s = sql.strip().rstrip(";").strip()
    s_chk = _sem_literais(s)
    if ";" in s_chk:
        raise PermissionError("Escrita bloqueada: envie uma instrução por vez.")
    verbo = (re.match(r"\s*(\w+)", s).group(1) if re.match(r"\s*(\w+)", s) else "").upper()
    if verbo in _VERBOS_PROIBIDOS:
        raise PermissionError(f"Comando '{verbo}' não é permitido neste notebook.")
    if verbo not in _VERBOS_ESCRITA:
        return                                                    # SELECT / SHOW / DESCRIBE / EXISTS ...
    prefixos = (PREFIXO_NOVO, *permitir_prefixos)
    if verbo == "RENAME":
        nomes = [_tabela_de(x) for par in _RENAME.findall(s_chk) for x in par]
    else:
        nomes = [_tabela_de(x) for x in _ALVO.findall(s_chk)]
    if not nomes:
        raise PermissionError(f"Escrita bloqueada: não identifiquei a tabela alvo em: {s[:80]}...")
    ruins = [n for n in nomes if not n.startswith(prefixos)]
    if ruins:
        raise PermissionError(f"Escrita bloqueada em {ruins}: só é permitido escrever em tabelas com prefixo {prefixos}.")


def executar(sql: str, permitir_prefixos=()) -> pd.DataFrame:
    validar_escrita(sql, permitir_prefixos)
    return _http(sql)


consultar = executar      # leituras também passam pela trava (e ela as libera)

# autoteste da trava (não toca no servidor)
_deve_passar = ["SELECT 1", "SHOW CREATE TABLE data2_portmap", f"CREATE TABLE {PREFIXO_NOVO}x AS data2_portmap",
                f"INSERT INTO {PREFIXO_NOVO}x SELECT * FROM data2_portmap LIMIT 1", f"DROP TABLE IF EXISTS `{PREFIXO_NOVO}x`",
                f"INSERT INTO {PREFIXO_NOVO}x VALUES ('idêntica a a; teste', 'DROP TABLE data_syn')"]
_deve_bloquear = ["INSERT INTO data2_portmap SELECT 1", "DROP TABLE data2_portmap", "DROP TABLE IF EXISTS data_tftp",
                  "TRUNCATE TABLE data_syn", "ALTER TABLE data_syn DROP COLUMN label", "CREATE TABLE outra AS data_syn",
                  "RENAME TABLE data_syn TO prep_syn", "CREATE DATABASE x", "DELETE FROM data_syn WHERE 1",
                  f"INSERT INTO {PREFIXO_NOVO}x SELECT 1; DROP TABLE data_syn", "SYSTEM DROP MARK CACHE", f"INSERT INTO {PREFIXO_NOVO}x VALUES ('a'); DROP TABLE data_syn"]
for _s in _deve_passar:
    validar_escrita(_s)
for _s in _deve_bloquear:
    try:
        validar_escrita(_s); raise AssertionError(f"a trava deixou passar: {_s}")
    except PermissionError:
        pass
print(f"Trava de segurança OK: {len(_deve_passar)} comandos permitidos e {len(_deve_bloquear)} bloqueados no autoteste.")

Trava de segurança OK: 6 comandos permitidos e 12 bloqueados no autoteste.


In [13]:
# ---------- ferramentas ----------
import math
import numpy as np
from datetime import datetime

def lit(s): return "'" + str(s).replace("\\", "\\\\").replace("'", "\\'") + "'"
def q(c): return "`" + str(c).replace("`", "``") + "`"

def consultar_num(sql):
    df = consultar(sql)
    for c in df.columns:
        try: df[c] = pd.to_numeric(df[c])
        except (ValueError, TypeError): pass
    return df

def n_linhas(tab, onde=""):
    return int(consultar_num(f"SELECT count() AS n FROM {tab}" + (f" WHERE {onde}" if onde else "")).n[0])

def existe(tab):
    return int(consultar_num(f"SELECT count() AS n FROM system.tables WHERE database = currentDatabase() AND name = {lit(tab)}").n[0]) > 0

def executar_robusto(sql, antes=None, tentativas=5, espera=120):
    # Escritas não são repetidas pelo transporte. Aqui, se o servidor estiver sem memória (Code 241) ou a conexão cair, esperamos e refazemos
    # (o comando `antes`, por exemplo um DROP, devolve o estado inicial; tudo aqui é idempotente).
    for k in range(1, tentativas + 1):
        try:
            if antes: executar(antes)
            return executar(sql)
        except PermissionError:
            raise
        except Exception as e:
            if isinstance(e, ErroConexao) or "241" in str(e) or "memory limit" in str(e).lower():
                print(f"  ⚠ falha por memória/conexão (tentativa {k}/{tentativas}): {str(e)[:160]}; aguardando {espera}s e refazendo...")
                time.sleep(espera); continue
            raise
    raise RuntimeError(f"não consegui concluir após {tentativas} tentativas: {sql[:100]}...")

def distribuir(total, tamanhos, alpha, tetos):
    # Reparte `total` linhas entre os tipos, com peso tamanho**alpha (alpha = 0: partes iguais; 1: proporcional), sem passar do teto de cada um.
    # O que sobra de um tipo pequeno (que atingiu o teto) é repartido entre os demais.
    q_ = {t: 0 for t in tamanhos}; livres = [t for t in tamanhos if tetos[t] > 0]; restante = int(total)
    if sum(tetos[t] for t in livres) < restante: raise ValueError(f"não há linhas suficientes: pedido {restante:,}, disponível {sum(tetos[t] for t in livres):,}")
    while restante > 0 and livres:
        w = {t: float(tamanhos[t]) ** alpha for t in livres}; sw = sum(w.values())
        alvo = {t: restante * w[t] / sw for t in livres}
        cheios = [t for t in livres if q_[t] + alvo[t] >= tetos[t]]
        if not cheios:
            base = {t: int(alvo[t]) for t in livres}; sobra = restante - sum(base.values())
            for t in sorted(livres, key=lambda t: alvo[t] - base[t], reverse=True)[:sobra]: base[t] += 1
            for t in livres: q_[t] += base[t]
            restante = 0
        else:
            for t in cheios:
                restante -= tetos[t] - q_[t]; q_[t] = tetos[t]; livres.remove(t)
    return q_

# autoteste de distribuir (não toca no servidor)
_a = distribuir(100, {"a": 1000, "b": 1000, "c": 1000, "d": 1000}, 0.0, {"a": 1000, "b": 1000, "c": 1000, "d": 1000}); assert _a == {"a": 25, "b": 25, "c": 25, "d": 25}, _a
_b = distribuir(100, {"a": 1000, "b": 1000, "c": 1000, "d": 10}, 0.0, {"a": 1000, "b": 1000, "c": 1000, "d": 10}); assert _b["d"] == 10 and sum(_b.values()) == 100 and set(_b[t] for t in "abc") <= {30, 31}, _b
_c = distribuir(100, {"a": 900, "b": 100}, 1.0, {"a": 900, "b": 100}); assert _c == {"a": 90, "b": 10}, _c
print("Autoteste da distribuição por tipo OK:", _b)

Autoteste da distribuição por tipo OK: {'a': 30, 'b': 30, 'c': 30, 'd': 10}


## 1. Parâmetros e o que existe

- `FRACAO_BENIGN_TREINO`: fração dos BENIGN (e, por tipo, dos ataques) que pode ir para o **ajuste** do modelo. O restante vira validação.
- `RAZAO_ATAQUE_BENIGNO`: 1,0 = balanceado. (Se um dia quiser 1 BENIGN : 3 ataques, use 3,0.)
- `ALPHA_ATAQUES`: como repartir os ataques entre os tipos. **0** = mesma quantidade por tipo (padrão); **1** = proporcional ao tamanho de cada tipo; **0,5** = intermediário.
- `VALIDACAO_ATAQUES`: quantas linhas de ataque terá a validação (amostra por tipo, com pesos).
- `SEMENTE`: fixa; a mesma seleção sai sempre. **Não a mude depois de ver resultados.**

A tabela abaixo mostra quantas linhas de cada tipo de ataque existem no treino (já sem duplicatas).

In [14]:
# ---------- 1.1 Parâmetros e levantamento ----------
SEMENTE = globals().get("SEMENTE", 20261018)
FRACAO_BENIGN_TREINO = globals().get("FRACAO_BENIGN_TREINO", 0.80)
RAZAO_ATAQUE_BENIGNO = globals().get("RAZAO_ATAQUE_BENIGNO", 1.0)
ALPHA_ATAQUES = globals().get("ALPHA_ATAQUES", 0.0)
VALIDACAO_ATAQUES = globals().get("VALIDACAO_ATAQUES", 50_000)
VIEW_BASE = "prep_vw_rf_bin_treino"                      # view definitiva do notebook 03
SEL_TAB, DADOS_TAB, V_TREINO, V_VALID = "prep_bal_bin_selecao", "prep_bal_bin_dados", "prep_vw_bal_bin_treino", "prep_vw_bal_bin_validacao"

faltam = [t for t in ("prep_final_treino", VIEW_BASE, "prep_vw_rf_bin_teste_sem_sobrep") if not existe(t)]
assert not faltam, f"Rode antes os notebooks 02 e 03: faltam {faltam}"
assert 0.5 <= FRACAO_BENIGN_TREINO < 1.0, "FRACAO_BENIGN_TREINO deve ficar entre 0,5 e 1 (precisa sobrar BENIGN para a validação)"

N_BENIGN = n_linhas("prep_final_treino", "label_binary = 0")
tp = consultar_num("SELECT label_multi AS tipo, count() AS n FROM prep_final_treino WHERE label_binary = 1 GROUP BY label_multi ORDER BY n DESC")
DISP = dict(zip(tp.tipo, tp.n.astype(int)))
print(f"BENIGN no treino: {N_BENIGN:,} | ataques: {sum(DISP.values()):,} em {len(DISP)} tipos")
display(tp.assign(nome=tp.tipo.str.replace(r"^DrDoS_", "", regex=True))[["nome", "n"]].rename(columns={"n": "linhas no treino"}).reset_index(drop=True))
print(f"\nPlano: {round(FRACAO_BENIGN_TREINO * N_BENIGN):,} BENIGN no ajuste + {round(RAZAO_ATAQUE_BENIGNO * round(FRACAO_BENIGN_TREINO * N_BENIGN)):,} ataques "
      f"= {round(FRACAO_BENIGN_TREINO * N_BENIGN) + round(RAZAO_ATAQUE_BENIGNO * round(FRACAO_BENIGN_TREINO * N_BENIGN)):,} linhas; validação: {N_BENIGN - round(FRACAO_BENIGN_TREINO * N_BENIGN):,} BENIGN + ~{VALIDACAO_ATAQUES:,} ataques (com pesos)")

BENIGN no treino: 50,768 | ataques: 8,206,575 em 12 tipos


,nome,linhas no treino
0,TFTP,4396919
1,NTP,1112936
2,UDP,1075337
3,SSDP,891221
4,MSSQL,207003
5,Syn,155515
6,SNMP,115316
7,DNS,113259
8,UDPLag,89066
9,LDAP,30099



Plano: 40,614 BENIGN no ajuste + 40,614 ataques = 81,228 linhas; validação: 10,154 BENIGN + ~50,000 ataques (com pesos)


## 2. A seleção (passo a passo)

1. **BENIGN.** Todos os BENIGN do treino são ordenados por um **número pseudoaleatório fixo** (`cityHash64(id_linha, SEMENTE)`). Os primeiros `FRACAO_BENIGN_TREINO` vão para o **ajuste**; os demais, para a **validação** (todos, com peso 1).
2. **Quantos ataques por tipo.** O total de ataques do ajuste é `RAZAO × (BENIGN do ajuste)`. Ele é repartido entre os tipos (partes iguais, por padrão). Cada tipo tem um **teto** (`FRACAO` das suas linhas), para sempre sobrar uma parte para a validação. Tipos pequenos que atingem o teto entram "cheios", e o que falta é repartido entre os outros.
3. **Sorteio dentro do tipo.** Em cada tipo, as linhas são ordenadas pelo mesmo número pseudoaleatório. As primeiras vão para o **ajuste** e as seguintes para a **validação**. Assim os dois grupos **nunca se repetem**, e cada um é um sorteio aleatório do tipo.
4. **Pesos da validação.** Cada ataque sorteado para a validação representa `(linhas restantes do tipo) ÷ (linhas sorteadas do tipo)` ataques reais. Esse é o seu peso. Os BENIGN da validação são só uma fração (20%) dos BENIGN, enquanto os ataques sorteados representam quase todos os ataques; por isso **cada BENIGN pesa mais que 1** (≈ 5 com os seus números), de modo que a validação ponderada tenha a **mesma proporção BENIGN:ataque do treino** (0,615%). Isso importa para **precisão, F1 e PR-AUC**; recall e taxa de falso alarme não dependem da proporção.

Para não ordenar milhões de linhas num servidor com pouca memória, o notebook primeiro pega, em cada tipo, um subconjunto aleatório um pouco maior que o necessário (filtro por `hash % 1.000.000`), e só ordena esse subconjunto.

In [15]:
# ---------- 2.1 Seleção ----------
def selecionar():
    h_ = lambda s: int(s)
    # 1) BENIGN: ordem pseudoaleatória fixa; primeiros -> ajuste, demais -> validação
    b = consultar(f"SELECT toString(id_linha) AS id, toString(cityHash64(id_linha, {SEMENTE})) AS h FROM prep_final_treino WHERE label_binary = 0")
    b = b.assign(hh=b.h.map(h_)).sort_values("hh", kind="stable").reset_index(drop=True)
    n_b_tr = int(round(FRACAO_BENIGN_TREINO * len(b)))
    linhas = [(i, "treino" if k < n_b_tr else "validacao", 0, "BENIGN", 1.0) for k, i in enumerate(b.id)]
    # 2) quantos ataques de cada tipo (ajuste e validação)
    teto_tr = {t: int(FRACAO_BENIGN_TREINO * n) for t, n in DISP.items()}
    n_att = int(round(RAZAO_ATAQUE_BENIGNO * n_b_tr))
    try:
        qtr = distribuir(n_att, DISP, ALPHA_ATAQUES, teto_tr)
    except ValueError as e:
        raise ValueError(f"{e}. Reduza RAZAO_ATAQUE_BENIGNO ou aumente FRACAO_BENIGN_TREINO.") from None
    resto = {t: DISP[t] - qtr[t] for t in DISP}
    val_total = min(int(VALIDACAO_ATAQUES), sum(resto.values()))
    qva = distribuir(val_total, resto, ALPHA_ATAQUES, resto)
    precisa = {t: qtr[t] + qva[t] for t in DISP if qtr[t] + qva[t] > 0}
    # 3) sorteio dentro de cada tipo, por subconjunto aleatório maior que o necessário (sem ordenar milhões de linhas)
    M, margem, sub = 1_000_000, 1.25, None
    for _ in range(4):
        K = {t: min(M, max(1, math.ceil(M * min(1.0, (margem * n + 6 * math.sqrt(n)) / DISP[t])))) for t, n in precisa.items()}
        tipos_sql, ks_sql = ", ".join(lit(t) for t in K), ", ".join(str(K[t]) for t in K)
        sub = consultar(f"SELECT toString(id_linha) AS id, label_multi AS tipo, toString(cityHash64(id_linha, {SEMENTE})) AS h FROM prep_final_treino "
                        f"WHERE label_binary = 1 AND cityHash64(id_linha, {SEMENTE}) % {M} < transform(label_multi, [{tipos_sql}], [{ks_sql}], 0)")
        cont = sub.groupby("tipo").size().to_dict()
        if all(cont.get(t, 0) >= n for t, n in precisa.items()): break
        margem *= 1.6
    else:
        raise RuntimeError("o subconjunto aleatório ficou menor que o necessário; aumente a margem")
    sub = sub.assign(hh=sub.h.map(h_))
    for t, g in sub.groupby("tipo"):
        g = g.sort_values("hh", kind="stable").reset_index(drop=True); nome = t.replace("DrDoS_", "", 1) if t.startswith("DrDoS_") else t
        for k in range(qtr[t]): linhas.append((g.id[k], "treino", 1, nome, 1.0))
        peso = resto[t] / qva[t] if qva[t] else 0.0
        for k in range(qtr[t], qtr[t] + qva[t]): linhas.append((g.id[k], "validacao", 1, nome, peso))
    df = pd.DataFrame(linhas, columns=["id", "papel", "y", "tipo", "peso"])
    # Peso dos BENIGN da validação: os ataques sorteados representam (quase) TODOS os ataques restantes, mas os BENIGN da validação são só uma fração dos BENIGN.
    # Para a validação ponderada manter a MESMA proporção BENIGN:ataque do treino (importa para precisão, F1 e PR-AUC), cada BENIGN pesa mais que 1.
    n_bv = int(((df.papel == "validacao") & (df.y == 0)).sum()); r_a = sum(resto[t] for t in DISP if qva[t] > 0)
    if n_bv and r_a: df.loc[(df.papel == "validacao") & (df.y == 0), "peso"] = (len(b) / sum(DISP.values())) * r_a / n_bv
    return df, qtr, qva, resto

SEL, QTR, QVA, RESTO = selecionar()
assert SEL.id.is_unique and SEL.id.map(str.isdigit).all(), "ids repetidos ou inválidos"
print(f"✔ Selecionadas {len(SEL):,} linhas: ajuste = {int(((SEL.papel == 'treino') & (SEL.y == 0)).sum()):,} BENIGN + {int(((SEL.papel == 'treino') & (SEL.y == 1)).sum()):,} ataques | "
      f"validação = {int(((SEL.papel == 'validacao') & (SEL.y == 0)).sum()):,} BENIGN + {int(((SEL.papel == 'validacao') & (SEL.y == 1)).sum()):,} ataques (com pesos)")

✔ Selecionadas 141,382 linhas: ajuste = 40,614 BENIGN + 40,614 ataques | validação = 10,154 BENIGN + 50,000 ataques (com pesos)


## 3. Gravação: auditoria, dados e views

1. A **tabela de auditoria** `prep_bal_bin_selecao` recebe as linhas escolhidas (papel, tipo, peso).
2. A tabela `prep_bal_bin_dados` copia, **da view definitiva** do notebook 03, os atributos dessas linhas. Assim valem as mesmas decisões: nulos pela mediana do treino e sem as colunas de AUC alto. Como a tabela é pequena, a leitura para o treino é rápida, e a seleção fica congelada.
3. As **views** `prep_vw_bal_bin_treino` e `prep_vw_bal_bin_validacao` mostram só `id_linha`, `y` (e `peso`, na validação) e os atributos. O `tipo` não aparece nelas.

Escritas grandes são refeitas automaticamente se o servidor estiver sem memória.

In [16]:
# ---------- 3.1 Gravação ----------
versao_bal = datetime.now().strftime("%Y%m%d_%H%M%S")
executar(f"DROP TABLE IF EXISTS `{SEL_TAB}`")
executar(f"CREATE TABLE {SEL_TAB} (id_linha UInt64, papel String, y UInt8, tipo String, peso Float64, versao String) ENGINE = MergeTree ORDER BY (papel, id_linha)")
for i in range(0, len(SEL), 20_000):
    pedaco = SEL.iloc[i:i + 20_000]
    executar_robusto(f"INSERT INTO {SEL_TAB} VALUES " + ", ".join(f"({r.id}, {lit(r.papel)}, {int(r.y)}, {lit(r.tipo)}, {float(r.peso)!r}, {lit(versao_bal)})" for r in pedaco.itertuples()))
print(f"✔ {SEL_TAB}: {n_linhas(SEL_TAB):,} linhas")

executar_robusto(f"CREATE TABLE {DADOS_TAB} ENGINE = MergeTree ORDER BY (papel, id_linha) AS "
                 f"SELECT s.papel AS papel, s.tipo AS tipo, s.peso AS peso, v.* FROM {VIEW_BASE} v INNER JOIN {SEL_TAB} s ON v.id_linha = s.id_linha SETTINGS max_threads = 2",
                 antes=f"DROP TABLE IF EXISTS `{DADOS_TAB}`")
print(f"✔ {DADOS_TAB}: {n_linhas(DADOS_TAB):,} linhas")

cols_base = consultar(f"SELECT name FROM system.columns WHERE database = currentDatabase() AND `table` = {lit(VIEW_BASE)} ORDER BY position")["name"].tolist()
cols_modelo = ", ".join(q(c) for c in cols_base if c not in ("id_linha", "y"))
executar("DROP VIEW IF EXISTS `prep_vw_bal_bin_refit`")        # sobra de uma versão anterior com refit (se existir); o refit fica para o notebook de treino
executar(f"DROP VIEW IF EXISTS `{V_TREINO}`")
executar(f"CREATE VIEW {V_TREINO} AS SELECT id_linha, y, {cols_modelo} FROM {DADOS_TAB} WHERE papel = 'treino'")
executar(f"DROP VIEW IF EXISTS `{V_VALID}`")
executar(f"CREATE VIEW {V_VALID} AS SELECT id_linha, y, peso, {cols_modelo} FROM {DADOS_TAB} WHERE papel = 'validacao'")
print(f"✔ Views: {V_TREINO} e {V_VALID}  | versão {versao_bal}")

✔ prep_bal_bin_selecao: 141,382 linhas
✔ prep_bal_bin_dados: 141,382 linhas
✔ Views: prep_vw_bal_bin_treino e prep_vw_bal_bin_validacao  | versão 20261018_204854


## 4. Validação

Confere que:
1. o treino está **balanceado** (mesma quantidade de BENIGN e de ataque) e as linhas de treino e de validação **não se repetem**;
2. todos os BENIGN que não foram para o ajuste estão na validação, os pesos dos ataques somam as linhas **restantes** de cada tipo, e os pesos dos BENIGN mantêm a **proporção BENIGN:ataque do treino**;
3. **todos os tipos de ataque** aparecem no ajuste, e a quantidade por tipo é a planejada;
4. os dados gravados são **idênticos** aos da view definitiva do notebook 03;
5. as views de treino **não expõem** `tipo`, `papel` nem rótulos, e não há nulos.

In [17]:
# ---------- 4.1 Validação ----------
ok = []
def verifica(nome, cond, detalhe=""):
    ok.append(bool(cond)); print(("✔" if cond else "✘"), nome, detalhe)

d = consultar_num(f"SELECT papel, y, count() AS n, sum(peso) AS peso FROM {SEL_TAB} GROUP BY papel, y")
g = {(r.papel, int(r.y)): (int(r.n), float(r.peso)) for r in d.itertuples()}
verifica("Treino balanceado: BENIGN = ataques × (1 / RAZAO)", g[("treino", 1)][0] == round(RAZAO_ATAQUE_BENIGNO * g[("treino", 0)][0]), f"({g[('treino', 0)][0]:,} BENIGN e {g[('treino', 1)][0]:,} ataques)")
verifica("Treino + validação de BENIGN = todos os BENIGN do treino", g[("treino", 0)][0] + g[("validacao", 0)][0] == N_BENIGN, f"({g[('treino', 0)][0]:,} + {g[('validacao', 0)][0]:,} = {N_BENIGN:,})")
verifica("Papéis disjuntos: nenhuma linha em treino e validação ao mesmo tempo", n_linhas(SEL_TAB) == int(consultar_num(f"SELECT uniqExact(id_linha) AS n FROM {SEL_TAB}").n[0]) == len(SEL))
verifica("Todas as linhas escolhidas existem em prep_final_treino com o rótulo certo",
         int(consultar_num(f"SELECT count() AS n FROM {SEL_TAB} s INNER JOIN prep_final_treino f ON s.id_linha = f.id_linha WHERE s.y = f.label_binary").n[0]) == len(SEL))
verifica("Pesos dos ataques da validação somam as linhas restantes de cada tipo", abs(g[("validacao", 1)][1] - sum(RESTO[t] for t in DISP if QVA[t] > 0)) < 1e-6 * max(1, g[("validacao", 1)][1]),
         f"(soma dos pesos {g[('validacao', 1)][1]:,.0f})")
razao_real, razao_val = N_BENIGN / sum(DISP.values()), g[("validacao", 0)][1] / g[("validacao", 1)][1]
verifica("Validação ponderada mantém a proporção BENIGN:ataque do treino", abs(razao_val - razao_real) < 1e-9 * max(1.0, razao_real),
         f"(BENIGN = {100 * razao_val / (1 + razao_val):.3f}% da validação ponderada; no treino: {100 * razao_real / (1 + razao_real):.3f}%; cada BENIGN pesa {g[('validacao', 0)][1] / g[('validacao', 0)][0]:.2f})")
por_tipo = consultar_num(f"SELECT tipo, papel, count() AS n FROM {SEL_TAB} WHERE y = 1 GROUP BY tipo, papel")
tr_t = {r.tipo: int(r.n) for r in por_tipo.itertuples() if r.papel == "treino"}
nome = lambda t: t.replace("DrDoS_", "", 1) if t.startswith("DrDoS_") else t
verifica("Todos os tipos de ataque aparecem no ajuste, com a quantidade planejada", all(tr_t.get(nome(t), 0) == QTR[t] for t in DISP) and all(QTR[t] > 0 for t in DISP), f"({len(tr_t)} tipos)")
verifica("Todo tipo com linhas restantes aparece na validação", all(any(r.tipo == nome(t) and r.papel == "validacao" for r in por_tipo.itertuples()) for t in DISP if RESTO[t] > 0))
dd = consultar_num(f"SELECT y, tipo = 'BENIGN' AS e_benign, count() AS n FROM {DADOS_TAB} GROUP BY y, e_benign")
verifica("y coerente com o tipo (y = 0 só para BENIGN)", all((int(r.y) == 0) == bool(r.e_benign) for r in dd.itertuples()))
verifica("Tabela de dados = tabela de seleção (mesmo número de linhas)", n_linhas(DADOS_TAB) == len(SEL), f"({n_linhas(DADOS_TAB):,})")

feats = [c for c in cols_base if c not in ("id_linha", "y")]
v = consultar(f"SELECT toString(id_linha) AS id, y, " + ", ".join(q(c) for c in feats) + f" FROM {DADOS_TAB} WHERE cityHash64(id_linha) % 20 = 0 ORDER BY id")
w = consultar(f"SELECT toString(id_linha) AS id, y, " + ", ".join(q(c) for c in feats) + f" FROM {VIEW_BASE} WHERE id_linha IN (SELECT id_linha FROM {DADOS_TAB} WHERE cityHash64(id_linha) % 20 = 0) ORDER BY id")
igual = len(v) == len(w) and (v["id"].values == w["id"].values).all() and np.allclose(v[feats].apply(pd.to_numeric).values.astype(float), w[feats].apply(pd.to_numeric).values.astype(float), rtol=1e-12, atol=1e-12)
verifica("Dados idênticos aos da view definitiva (amostra)", igual and len(v) > 0, f"({len(v):,} linhas comparadas)")

proibidas = {"tipo", "papel", "label_original", "label_multi", "label_binary", "origem_tabela", "n_repeticoes", "conflito_rotulo", "conflito_binario", "sobrepoe_treino", "b"}
for vw, esp in ((V_TREINO, ["id_linha", "y"] + feats), (V_VALID, ["id_linha", "y", "peso"] + feats)):
    cv = consultar(f"SELECT name FROM system.columns WHERE database = currentDatabase() AND `table` = {lit(vw)} ORDER BY position")["name"].tolist()
    verifica(f"{vw}: colunas esperadas e sem tipo/papel/rótulos", cv == esp and not (proibidas & set(cv)), f"({len(cv)} colunas)")
    nulos = consultar_num(f"SELECT " + ", ".join(f"countIf(isNull({q(c)})) AS {q(c)}" for c in feats) + f" FROM {vw}").iloc[0]
    verifica(f"{vw}: sem nulos nos atributos", int(nulos.sum()) == 0)
assert all(ok), "há checagens reprovadas acima"
print(f"\n✔ {sum(ok)}/{len(ok)} checagens aprovadas.")

✔ Treino balanceado: BENIGN = ataques × (1 / RAZAO) (40,614 BENIGN e 40,614 ataques)
✔ Treino + validação de BENIGN = todos os BENIGN do treino (40,614 + 10,154 = 50,768)
✔ Papéis disjuntos: nenhuma linha em treino e validação ao mesmo tempo 
✔ Todas as linhas escolhidas existem em prep_final_treino com o rótulo certo 
✔ Pesos dos ataques da validação somam as linhas restantes de cada tipo (soma dos pesos 8,165,961)
✔ Validação ponderada mantém a proporção BENIGN:ataque do treino (BENIGN = 0.615% da validação ponderada; no treino: 0.615%; cada BENIGN pesa 4.98)
✔ Todos os tipos de ataque aparecem no ajuste, com a quantidade planejada (12 tipos)
✔ Todo tipo com linhas restantes aparece na validação 
✔ y coerente com o tipo (y = 0 só para BENIGN) 
✔ Tabela de dados = tabela de seleção (mesmo número de linhas) (141,382)
✔ Dados idênticos aos da view definitiva (amostra) (7,048 linhas comparadas)
✔ prep_vw_bal_bin_treino: colunas esperadas e sem tipo/papel/rótulos (45 colunas)
✔ prep_vw_ba

## 5. Resumo e leitura

Mostra, **por tipo de ataque**, quantas linhas existem, quantas foram para o ajuste e para a validação, e o peso. Mostra também o teste, que **não é balanceado**: a proporção de BENIGN dele é a "linha de base" para interpretar a precisão e o PR-AUC.

Duas informações para o TCC:
- **Linhas em conflito** (`conflito_binario = 1`): as mesmas características aparecem como BENIGN e como ataque. Nenhum modelo acerta as duas. Aqui só são **contadas**, e nenhuma é removida.
- **Tamanho do SVM:** o número de linhas do ajuste é o que decide o tempo do SVM com kernel. Antes de treinar, vale medir o tempo com 5 mil, 10 mil e 20 mil linhas (como combinamos) e confirmar que o conjunto inteiro cabe no seu tempo.

`ler_bal_bin("treino")` e `ler_bal_bin("validacao")` leem os conjuntos inteiros (são pequenos).

In [18]:
# ---------- 5.1 Resumo e leitura ----------
r = consultar_num(f"SELECT tipo, papel, count() AS n, avg(peso) AS peso FROM {SEL_TAB} GROUP BY tipo, papel")
t = r.pivot_table(index="tipo", columns="papel", values="n", aggfunc="sum", fill_value=0).astype(int)
t["no treino (total)"] = pd.Series({nome(k): v for k, v in DISP.items()}); t["no treino (total)"] = t["no treino (total)"].fillna(N_BENIGN).astype(int)
t["peso médio na validação"] = r[r.papel == "validacao"].set_index("tipo")["peso"].round(1)
t = t.rename(columns={"treino": "ajuste", "validacao": "validação (linhas)"})[["no treino (total)", "ajuste", "validação (linhas)", "peso médio na validação"]]
print("== Linhas por tipo =="); display(t.sort_values("ajuste", ascending=False))

teste = consultar_num("SELECT y, count() AS n FROM prep_vw_rf_bin_teste_sem_sobrep GROUP BY y ORDER BY y")
pb = teste.n[teste.y == 0].iloc[0] / teste.n.sum() * 100
print(f"\nTeste (NÃO balanceado, como está): {int(teste.n[teste.y == 0].iloc[0]):,} BENIGN e {int(teste.n[teste.y == 1].iloc[0]):,} ataques → BENIGN = {pb:.2f}% (esta é a linha de base do PR-AUC do BENIGN)")
cf = n_linhas("prep_final_treino", f"conflito_binario = 1 AND id_linha IN (SELECT id_linha FROM {SEL_TAB} WHERE papel = 'treino')")
print(f"Linhas em conflito BENIGN × ataque dentro do ajuste: {cf:,} (apenas contadas; nenhuma foi removida)")
print(f"RAM do ajuste em float64: {n_linhas(DADOS_TAB, chr(112) + 'apel = ' + lit('treino')) * len(feats) * 8 / 2**20:.1f} MiB")

def ler_bal_bin(papel="treino"):
    # papel: 'treino' (ajuste balanceado) ou 'validacao' (com a coluna peso). Os conjuntos são pequenos: leitura inteira.
    return consultar_num(f"SELECT * FROM {V_TREINO if papel == 'treino' else V_VALID}")

for p in ("treino", "validacao"):
    a = ler_bal_bin(p); assert not a.drop(columns=["id_linha"]).isna().any().any()
    print(f"Leitura de teste ({p}): {a.shape[0]:,} linhas x {a.shape[1]} colunas, sem NaN | classes: {a['y'].value_counts().to_dict()}")

== Linhas por tipo ==


papel,no treino (total),ajuste,validação (linhas),peso médio na validação
tipo,,,,
BENIGN,50768,40614,10154,5.0
TFTP,4396919,3663,4538,968.1
DNS,113259,3662,4538,24.2
MSSQL,207003,3662,4538,44.8
LDAP,30099,3662,4538,5.8
NTP,1112936,3662,4538,244.4
NetBIOS,19490,3662,4537,3.5
SSDP,891221,3662,4538,195.6
SNMP,115316,3662,4538,24.6



Teste (NÃO balanceado, como está): 56,216 BENIGN e 7,162,973 ataques → BENIGN = 0.78% (esta é a linha de base do PR-AUC do BENIGN)
Linhas em conflito BENIGN × ataque dentro do ajuste: 18 (apenas contadas; nenhuma foi removida)
RAM do ajuste em float64: 26.6 MiB
Leitura de teste (treino): 81,228 linhas x 45 colunas, sem NaN | classes: {1: 40614, 0: 40614}
Leitura de teste (validacao): 60,154 linhas x 46 colunas, sem NaN | classes: {1: 50000, 0: 10154}
